# Chunk-Level XGBoost — Random Slopes: Quantity vs Weight vs Cube vs Hour

Loops over four slope candidates separately to find which produces the most
meaningful worker heterogeneity at the chunk level.

For each candidate the random slopes model is:
`Time_Delta_sec ~ SlopeVar_z + (1 + SlopeVar_z | UserID)`

**Hour motivation:** as workers progress through their shift, fatigue may accumulate
and pick times may increase. If some workers fatigue faster than others, a random
slope on `hour` captures this. `hour` (0-23) is used as a continuous ordered variable
rather than the bucketed `time_of_day` already in the feature set.

**Models compared at chunk level per WorkCode:**
1. **Baseline** — no worker features
2. **+ Intercept** — random intercept only (worker overall speed)
3. **+ Intercept + Slope[Quantity]**
4. **+ Intercept + Slope[Weight]**
5. **+ Intercept + Slope[Cube]**
6. **+ Intercept + Slope[Hour]**

Baseline and + Intercept are run once per WorkCode. Each slope is run separately.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
from feature_engineer import get_engineered_df

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

PATH             = Path("../data/processed")
WAREHOUSE        = "OE"
WORKCODES        = ["10", "20", "30"]
MAX_TIME         = 300
BLOCK_SIZE       = 50
RANDOM_STATE     = 2026
SLOPE_CANDIDATES = ["Quantity", "Weight", "Cube", "Hour"]

## Helper Functions

In [ ]:
def load_engineered_data(warehouse, workcode, max_time=300):
    d, features_all, cat_cols_all = get_engineered_df(
        file_path=PATH / f"{warehouse.lower()}_detailed.parquet",
        warehouse=warehouse,
        max_time=max_time,
        work_code=str(workcode)
    )
    d = d.copy()
    d["Timestamp"] = pd.to_datetime(d["Timestamp"], errors="coerce")
    d = d.dropna(subset=["Timestamp"]).copy()
    d["date"]     = d["Timestamp"].dt.date
    d["WorkCode"] = d["WorkCode"].astype(str).str.replace(".0", "", regex=False)

    # Raw hour column for the fatigue slope — kept separate from the
    # bucketed time_of_day feature already in the engineered feature set
    d["Hour"] = d["Timestamp"].dt.hour.astype(float)

    distance_num = ["Travel_Distance", "log_travel_distance"]
    distance_cat = ["same_aisle", "same_location", "diff_level"]
    features = [f for f in features_all if f not in distance_num + distance_cat]
    cat_cols = [c for c in cat_cols_all if c not in distance_cat]

    return d, features, cat_cols


def split_by_days(df, test_ratio=0.15):
    all_days  = sorted(df["date"].dropna().unique())
    n_test    = max(1, int(round(len(all_days) * test_ratio)))
    test_days = all_days[-n_test:]
    train_df  = df[df["date"] < test_days[0]].copy()
    test_df   = df[df["date"].isin(test_days)].copy()
    return train_df, test_df


def make_X(train_df, test_df, features, cat_cols):
    X_train = pd.get_dummies(train_df[features], columns=cat_cols, drop_first=True)
    X_test  = pd.get_dummies(test_df[features],  columns=cat_cols, drop_first=True)
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    return X_train, X_test


def make_test_blocks(test_df, block_size=50):
    d = test_df.sort_values(["UserID", "Timestamp"]).copy()
    blocks = []
    for (uid, day), g in d.groupby(["UserID", "date"], sort=False):
        g = g.reset_index()
        for start in range(0, len(g) - block_size + 1, block_size):
            chunk = g.iloc[start:start + block_size]
            if len(chunk) == block_size:
                blocks.append({
                    "BlockID": f"{uid}_{day}_{start // block_size}",
                    "UserID":  uid,
                    "date":    day,
                    "indices": chunk["index"].tolist()
                })
    return blocks


def eval_blocks(blocks, actual_series, pred_series):
    actual_b = [actual_series.loc[b["indices"]].sum() for b in blocks]
    pred_b   = [pred_series.loc[b["indices"]].sum()   for b in blocks]
    return mean_absolute_error(actual_b, pred_b), r2_score(actual_b, pred_b)

## Random Effects Estimation Functions

In [ ]:
def estimate_intercept_only(train_df):
    """Random intercept: Time_Delta_sec ~ 1 + (1|UserID)"""
    df_re = train_df[["UserID", "Time_Delta_sec"]].dropna().copy()
    if df_re["UserID"].nunique() < 2:
        return pd.DataFrame({"UserID": df_re["UserID"].unique(), "worker_intercept": 0.0})

    result = smf.mixedlm(
        "Time_Delta_sec ~ 1", data=df_re, groups=df_re["UserID"]
    ).fit(reml=True, disp=False)

    icc = result.cov_re.values[0][0] / (result.cov_re.values[0][0] + result.scale)
    print(f"    [Intercept] Grand mean: {result.fe_params['Intercept']:.1f}s | "
          f"Worker SD: {np.sqrt(result.cov_re.values[0][0]):.1f}s | ICC: {icc:.3f}")

    return pd.DataFrame({
        "UserID":           list(result.random_effects.keys()),
        "worker_intercept": [float(v.iloc[0]) for v in result.random_effects.values()]
    })


def estimate_random_slopes(train_df, slope_var):
    """
    Fits: Time_Delta_sec ~ slope_var_z + (1 + slope_var_z | UserID)
    The slope variable is standardised before fitting for convergence stability.
    Returns per-worker intercept AND slope.
    Falls back to intercept-only and slope=0 on convergence failure.
    Works for any slope_var in SLOPE_CANDIDATES including Hour.
    """
    slope_col = f"worker_slope_{slope_var.lower()}"
    df_re = train_df[["UserID", "Time_Delta_sec", slope_var]].dropna().copy()

    if df_re["UserID"].nunique() < 2:
        return pd.DataFrame({
            "UserID": df_re["UserID"].unique(),
            "worker_intercept": 0.0,
            slope_col: 0.0
        })

    mu = df_re[slope_var].mean()
    sd = df_re[slope_var].std() + 1e-8
    df_re["sv_z"] = (df_re[slope_var] - mu) / sd

    try:
        result = smf.mixedlm(
            "Time_Delta_sec ~ sv_z",
            data=df_re,
            groups=df_re["UserID"],
            exog_re=df_re[["sv_z"]]
        ).fit(reml=True, disp=False)

        rows = [{
            "UserID":           uid,
            "worker_intercept": float(re.iloc[0]),
            slope_col:          float(re.iloc[1]) if len(re) > 1 else 0.0
        } for uid, re in result.random_effects.items()]

        slope_sd = np.sqrt(max(result.cov_re.values[1][1], 0)) if result.cov_re.shape[0] > 1 else 0.0
        print(f"    [{slope_var} slope] Fixed effect: "
              f"{result.fe_params.get('sv_z', float('nan')):.3f}s per SD | "
              f"Slope SD across workers: {slope_sd:.3f}")

        return pd.DataFrame(rows)

    except Exception as e:
        print(f"    [{slope_var} slope] Failed ({e}) — using intercept only")
        fallback = estimate_intercept_only(train_df)
        fallback[slope_col] = 0.0
        return fallback

## Main Loop — All Slope Candidates × All WorkCodes

Outer loop: slope candidate (Quantity, Weight, Cube, Hour)  
Inner loop: WorkCode (10, 20, 30)  
Baseline and + Intercept are only run once per WorkCode to avoid redundancy.

In [ ]:
all_results        = []
slope_dist_data    = {}   # (slope_var, wc) -> worker RE dataframe for distribution plots
recorded_baselines = set()  # avoid re-running baseline/intercept for each slope candidate

xgb_params = dict(
    n_estimators=800, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1
)

for slope_var in SLOPE_CANDIDATES:
    print(f"\n{'#'*60}")
    print(f"SLOPE CANDIDATE: {slope_var}")
    print(f"{'#'*60}")
    slope_col = f"worker_slope_{slope_var.lower()}"

    for wc in WORKCODES:
        print(f"\n  {'='*50}")
        print(f"  WorkCode {wc}")
        print(f"  {'='*50}")

        df_wc, features, cat_cols = load_engineered_data(WAREHOUSE, wc, MAX_TIME)
        train_df, test_df = split_by_days(df_wc)
        print(f"  Train: {len(train_df)} | Test: {len(test_df)}")

        # Fit random slopes model (always returns intercept + slope columns)
        slopes_df = estimate_random_slopes(train_df, slope_var)
        slope_dist_data[(slope_var, wc)] = slopes_df

        # Merge worker features
        train_df = train_df.merge(slopes_df, on="UserID", how="left")
        test_df  = test_df.merge(slopes_df,  on="UserID", how="left")
        for col in ["worker_intercept", slope_col]:
            train_df[col] = train_df[col].fillna(0.0)
            test_df[col]  = test_df[col].fillna(0.0)

        train_df = train_df.reset_index(drop=True)
        test_df  = test_df.reset_index(drop=True)
        y_train  = train_df["Time_Delta_sec"].astype(float)
        y_test   = test_df["Time_Delta_sec"].astype(float)

        cats_clean  = [c for c in cat_cols  if c != "efficient_user"]
        feats_clean = [f for f in features  if f != "efficient_user"]

        scenarios = {
            "Baseline":                          feats_clean,
            "+ Intercept":                       feats_clean + ["worker_intercept"],
            f"+ Intercept + Slope[{slope_var}]": feats_clean + ["worker_intercept", slope_col],
        }

        blocks = make_test_blocks(test_df, block_size=BLOCK_SIZE)

        for label, feats in scenarios.items():
            # Only run Baseline and + Intercept once per WorkCode
            key = (wc, label)
            if label in ["Baseline", "+ Intercept"] and key in recorded_baselines:
                continue

            X_train, X_test = make_X(train_df, test_df, feats, cats_clean)
            model = XGBRegressor(**xgb_params)
            model.fit(X_train, y_train)
            preds = pd.Series(model.predict(X_test), index=test_df.index)

            mae, r2 = eval_blocks(blocks, y_test, preds)
            all_results.append({
                "SlopeCandidate": slope_var if "Slope" in label else "—",
                "WorkCode":       wc,
                "Model":          label,
                "n_blocks":       len(blocks),
                "mae_per_task":   round(mae / BLOCK_SIZE, 3),
                "r2":             round(r2, 4),
            })

            if label in ["Baseline", "+ Intercept"]:
                recorded_baselines.add(key)

results_df = pd.DataFrame(all_results)
print("\nDone.")

## Results Table

In [ ]:
print(f"Warehouse: {WAREHOUSE} | Block size: {BLOCK_SIZE} tasks\n")
display(
    results_df
    .sort_values(["WorkCode", "mae_per_task"])
    .reset_index(drop=True)
)

## Summary — Which Slope Candidate Wins per WorkCode?

In [ ]:
summary_rows = []
for wc in WORKCODES:
    wc_df    = results_df[results_df["WorkCode"] == wc]
    mae_base = wc_df[wc_df["Model"] == "Baseline"]["mae_per_task"].values
    mae_int  = wc_df[wc_df["Model"] == "+ Intercept"]["mae_per_task"].values
    if len(mae_base) == 0:
        continue
    mae_base = mae_base[0]
    mae_int  = mae_int[0] if len(mae_int) > 0 else float("nan")

    for sv in SLOPE_CANDIDATES:
        label = f"+ Intercept + Slope[{sv}]"
        row   = wc_df[wc_df["Model"] == label]["mae_per_task"].values
        mae_s = row[0] if len(row) > 0 else float("nan")
        summary_rows.append({
            "WorkCode":                  wc,
            "Slope":                     sv,
            "MAE/task Baseline (s)":     mae_base,
            "MAE/task + Intercept (s)":  mae_int,
            "MAE/task + Slope (s)":      mae_s,
            "Gain vs Baseline (s)":      round(mae_base - mae_s, 3),
            "Gain vs Intercept (s)":     round(mae_int  - mae_s, 3),
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print("\nBest slope candidate per WorkCode:")
for wc in WORKCODES:
    sub = summary_df[summary_df["WorkCode"] == wc]
    if sub.empty:
        continue
    best = sub.loc[sub["MAE/task + Slope (s)"].idxmin()]
    print(f"  WC{wc}: {best['Slope']} "
          f"— MAE/task {best['MAE/task + Slope (s)']:.3f}s "
          f"(gain vs intercept: {best['Gain vs Intercept (s)']:.3f}s)")

## Slope Distribution Plots — WC30 (Most Data)

Wider spread = workers genuinely differ on that dimension.  
Tight spike around zero = slope adds no meaningful worker heterogeneity.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, sv in zip(axes, SLOPE_CANDIDATES):
    col  = f"worker_slope_{sv.lower()}"
    data = slope_dist_data.get((sv, "30"))
    if data is None or col not in data.columns:
        ax.set_title(f"{sv} — no data")
        continue
    slope_sd = data[col].std()
    ax.hist(data[col], bins=15, edgecolor="black", color="steelblue", alpha=0.8)
    ax.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Average slope")
    ax.set_title(f"Worker slopes on {sv} (WC30)\nSD = {slope_sd:.3f}")
    ax.set_xlabel(f"Slope shift (per std dev of {sv})")
    ax.set_ylabel("Number of workers")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Wider spread = workers genuinely differ on that feature | "
    "Tight spike at 0 = no meaningful heterogeneity",
    fontsize=11
)
plt.tight_layout()
plt.show()

## Fixed Effect of Hour — Does the Average Worker Slow Down Over the Day?

Even if workers don't *differ* in their fatigue slope (slope SD ≈ 0),
there may still be a population-level trend — everyone slows down later in the shift.
This bar chart shows mean pick time per hour on WC30 training data.

In [ ]:
df_wc30, _, _ = load_engineered_data(WAREHOUSE, "30", MAX_TIME)
train30, _    = split_by_days(df_wc30)

hour_means = (
    train30.groupby("Hour")["Time_Delta_sec"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "Mean pick time (s)", "count": "N picks"})
)
print("WC30 — Mean pick time by hour of day (training set):\n")
print(hour_means.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hour_means["Hour"], hour_means["Mean pick time (s)"],
       color="steelblue", edgecolor="black", alpha=0.8)
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Mean Pick Time (s)")
ax.set_title("WC30 — Mean Pick Time by Hour of Day (Training Set)")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Actual vs Predicted — Best Slope Model per WorkCode (WC20 & WC30)

In [ ]:
# Re-fit and predict using the best slope candidate per WorkCode
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, wc in zip(axes, ["20", "30"]):
    sub    = summary_df[summary_df["WorkCode"] == wc]
    sv     = sub.loc[sub["MAE/task + Slope (s)"].idxmin(), "Slope"]
    slope_col = f"worker_slope_{sv.lower()}"

    df_wc, features, cat_cols = load_engineered_data(WAREHOUSE, wc, MAX_TIME)
    train_df, test_df = split_by_days(df_wc)

    slopes_df = estimate_random_slopes(train_df, sv)
    train_df  = train_df.merge(slopes_df, on="UserID", how="left")
    test_df   = test_df.merge(slopes_df,  on="UserID", how="left")
    for col in ["worker_intercept", slope_col]:
        train_df[col] = train_df[col].fillna(0.0)
        test_df[col]  = test_df[col].fillna(0.0)
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)
    y_train  = train_df["Time_Delta_sec"].astype(float)
    y_test   = test_df["Time_Delta_sec"].astype(float)

    cats_clean  = [c for c in cat_cols  if c != "efficient_user"]
    feats_clean = [f for f in features  if f != "efficient_user"]
    feats_plot  = feats_clean + ["worker_intercept", slope_col]

    X_train, X_test = make_X(train_df, test_df, feats_plot, cats_clean)
    model  = XGBRegressor(**xgb_params)
    model.fit(X_train, y_train)
    preds  = pd.Series(model.predict(X_test), index=test_df.index)
    blocks = make_test_blocks(test_df, block_size=BLOCK_SIZE)

    actual_b = [y_test.loc[b["indices"]].sum() for b in blocks]
    pred_b   = [preds.loc[b["indices"]].sum()   for b in blocks]

    ax.scatter(actual_b, pred_b, alpha=0.6, s=30)
    lims = [min(min(actual_b), min(pred_b)), max(max(actual_b), max(pred_b))]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual total time (s) per block")
    ax.set_ylabel("Predicted total time (s) per block")
    ax.set_title(f"WC{wc} — Best slope: {sv}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Warehouse {WAREHOUSE} | Block size = {BLOCK_SIZE} tasks", fontsize=13)
plt.tight_layout()
plt.show()